# 03 — DL com Wavelet Fixa (db4)

DWT db4 **fixa** como front-end antes do backbone. Mesmo filtro para todos os canais.

**Recomendado:** rodar todos os modelos/configs via fila GPU multi-processo:
```bash
cd tests/uwave-gesture && MODES=db4 python run_dl_queue.py --fresh
```
Abaixo, um exemplo **inline** (1 modelo) para inspeção rápida.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '.')
sys.path.insert(0, 'config')
from experiment_config import (
    UWAVE_CONFIG, DATA_DIR, RESULTS_DIR, SEED,
    WAVELET_CONFIG, LEARNED_WAVELET_CONFIG, DL_TRAINING_CONFIG,
    ML_MODELS_CONFIG, ML_SEARCH_CONFIG, build_param_dist,
)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(SEED)
print('Dataset:', UWAVE_CONFIG['dataset_name'], '| classes:', UWAVE_CONFIG['n_classes'],
      '| seq_len:', UWAVE_CONFIG['sequence_length'], '| canais:', UWAVE_CONFIG['n_features'])

In [ ]:
from src.data_loader import UWaveDataLoader
from src.pipeline_queue import UWaveExperimentPipeline
import experiment_config as C

MODE = 'db4'
MODEL = 'CNN'
if MODE in ('raw', 'db4'):
    cfg = {**C.DL_MODELS_CONFIG[MODEL], **C.DL_TRAINING_CONFIG}
    if MODE == 'db4':
        cfg.update({k: C.LEARNED_WAVELET_CONFIG[k] for k in ('levels','align')})
else:
    cfg = {**C.LEARNED_WAVELET_MODELS_CONFIG[MODEL], **C.DL_TRAINING_CONFIG, **C.LEARNED_WAVELET_CONFIG}
cfg['epochs'] = 10  # exemplo rápido; a fila usa C.DL_TRAINING_CONFIG['epochs']

pipe = UWaveExperimentPipeline(MODEL, MODE, cfg, config_idx=0,
                               results_dir=str(RESULTS_DIR / '_notebook_demo'), data_dir=str(DATA_DIR))
metrics = pipe.run()
print({k: round(v,4) for k,v in metrics.items() if k.startswith('test_')})